# Uvicorn/JupyUvi helpers

In [ ]:
#| default_exp server

In [ ]:
# await greet('pote.server')

<!-- linkedto: _21a088dd -->
Solveit version: **0.0.85**  
dialoghelper version: **0.2.16**  
git branch: **dev-log**  
git changes: [' M ../dutil/hx_bridge.js', ' M ../explorer/htmx_bridge.ipynb', ' M 00_htmx_bridge.ipynb', '?? ../explorer/.sesskey']

Hi! I'm Vic, we're going to work together on the project **pote.server**.

In [ ]:
#| export
import asyncio, logging, subprocess
from contextlib import asynccontextmanager
from datetime import datetime
from functools import partial
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, JupyUviAsync, is_port_free, HTMX
from fasthtml.xtend import Script, Link
from dialoghelper.tmux import shell_ret
from pote.dutil import set_var

In [ ]:
import os, json
from fastcore.foundation import store_attr, patch
from fastcore.test import *
from pyskills import doc
from pyskills import *
from fasthtml.components import Div, Span
from dialoghelper.core import add_msg, iife_a, add_html_a, read_msg, add_scr_a, js_run_a, js_eval_a
from pote.logger import logger
from pote.dutil import link_msg, wait_for_a

In [ ]:
# from inspect import getsource
# await add_msg(getsource(fasthtml.jupyter))

Please see above message with the source code for `fasthtml.jupyter`. What does it do? Explain.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

## helpers

In [ ]:
#| export
def kill_port(port): subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
#| export
def is_serving(srv, timeout=1):
    "Check if a JupyUvi instance is live and serving requests"
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        return httpx.get(f"http://localhost:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

In [ ]:
#| export
def wrap_endp(response):
    "If `response` is not callable, wrap a str/FT/sequence str|FT as a async endpoint"
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
#| export
def find_server(port=8000):
    "Scan IPython user_ns for a JupyUvi instance on `port`"
    import IPython
    ns = IPython.get_ipython().user_ns
    for k,v in ns.items():
        if isinstance(v, JupyUvi) and v.port == port and v.server and v.server.started and not v.server.should_exit: return v,k
    return None, None

In [ ]:
test_eq(find_server(), (None,None))

## server

In [ ]:
#| export
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
#| export
def get_preview(app): return partial(HTMX, app=app)

def ensure_server(srv=None, port=8000, app=None, hdrs=None):
    if not srv: srv,_ = find_server(port)
    if not srv:
        app = app if app else FastHTML(hdrs=hdrs if hdrs is not None else daisyui_hdrs)
        srv = JupyUvi(app)
    else: app = srv.app
    return app, app.route, srv, get_preview(app)

def stop_server(port=8000):
    srv, sym = find_server(port)
    if is_serving(srv):
        srv.stop()
        set_var(sym, None)

In [ ]:
kill_port(8000)

In [ ]:
app, rt, srv, p = ensure_server()
test_is(srv is not None, True)

In [ ]:
app2, _, srv2, _ = ensure_server()
test_eq((app,srv), (app2,srv2))

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
stop_server()

In [ ]:
test_is(globals()['srv'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| exporti
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record): self.log(f"{(record.levelname+':'):<10}{self.format(record)}")

In [ ]:
#| export
def setup_uvi_logging(_log=None):
    _log = _log or log
    _lh = LoggerHandler(_log)
    for name in ('uvicorn', 'uvicorn.error', 'uvicorn.access'):
        lg = logging.getLogger(name)
        lg.handlers, lg.propagate = [_lh], False
        lg.setLevel(logging.DEBUG)

In [ ]:
# @asynccontextmanager
# async def local_server(app=None, host='0.0.0.0', port=8000, quiet=False, timeout=3, **uvikw):
#     "Run a Starlette app in a background thread for testing"
#     if not is_port_free(port, host): raise RuntimeError(f"Port {port} already in use")
#     app = app or Starlette()
#     srv = uvicorn.Server(uvicorn.Config(app, host=host, port=port, **uvikw))
#     thrd = threading.Thread(target=srv.run, daemon=True)
#     thrd.start()
#     deadline = (loop := asyncio.get_running_loop()).time() + timeout
#     while not srv.started:
#         if loop.time() > deadline: raise RuntimeError(f"Server failed to start on port {port} within {timeout}s")
#         await asyncio.sleep(0.01)
#     if not quiet:
#         try: print(f">>>{thrd.is_alive()=}\n{shell_ret(f'lsof -i :{port}')}")
#         except Exception as e: print(f">>>startup diag failed: {e}")
#     try: yield thrd, srv, app
#     finally:
#         srv.should_exit = True
#         thrd.join(timeout=timeout)
#         if not quiet:
#             if thrd.is_alive(): print(f"WARNING: server thread still alive after {timeout}s")
#             try: print(f"<<<{thrd.is_alive()=}\n{shell_ret(f'lsof -i :{port}')}")
#             except Exception as e: print(f"<<<teardown diag failed: {e}")

In [ ]:
# @asynccontextmanager
# async def local_server(port=8000, **kw):
#     "Yield a running JupyUvi on `port`, spinning one up if needed"
#     server = None
#     if not is_port_free(port):
#         srv = find_server(port)
#         if srv is None: raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
#     else: server = srv = JupyUvi(FastHTML(), port=port, **kw)
#     try: yield srv, srv.app, srv.app.route
#     finally:
#         if server: server.stop()

In [ ]:
#| export
@asynccontextmanager
async def local_server(app=None, port=8000, after_start=None, before_stop=None, quiet=True, timeout=5, **jupyuvi_kw):
    "Yield a running JupyUvi server that can be used for testing; spins one up if needed, then cleanly tears it down afterwards"
    server = None
    if not is_port_free(port): 
        srv,_ = find_server(port)
        if srv is None: raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
        if not is_serving(srv): raise RuntimeError(f"Port {port} in use but JupyUvi server is not usable")
    else: 
        server = srv = JupyUviAsync(app or FastHTML(), port=port, **jupyuvi_kw)
        await srv.start()
        if not quiet:
            try: print(f">>> started: {srv.server.started}\n{shell_ret(f'lsof -i :{port}')}")
            except Exception as e: print(f">>> server startup failed: {e}")
    if after_start: await after_start(srv) if asyncio.iscoroutinefunction(after_start) else after_start(srv)
    try: yield srv
    finally:
        if before_stop: await before_stop(srv) if asyncio.iscoroutinefunction(before_stop) else before_stop(srv)
        if not server: return
        srv.server.should_exit = True
        deadline = (loop := asyncio.get_running_loop()).time() + timeout
        while not is_port_free(port):
            if loop.time() > deadline:
                print(f"WARNING: port {port} still in use after {timeout}s"); break
            await asyncio.sleep(0.1)
        if not quiet:
            try: print(f"<<< after stop: port {port} {'still in use' if shell_ret(f'lsof -i :{port}') else 'free'}")
            except Exception as e: print(f"<<< teardown failed: {e}")

In [ ]:
setup_uvi_logging()
await log.show(True)

[09:49:03.589] INFO:     Started server process [7523]
[09:49:03.633] INFO:     Waiting for application startup.
[09:49:03.685] TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}
[09:49:03.728] TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}
[09:49:03.770] TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}
[09:49:03.816] INFO:     Application startup complete.
[09:49:03.859] INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
[09:49:04.002] INFO:     Shutting down
[09:49:04.148] INFO:     Waiting for application shutdown.
[09:49:04.196] TRACE:    ASGI [1] Receive {'type': 'lifespan.shutdown'}
[09:49:04.239] TRACE:    ASGI [1] Send {'type': 'lifespan.shutdown.complete'}
[09:49:04.282] TRACE:    ASGI [1] Completed
[09:49:04.325] INFO:     Application shutdown complete.
[09:49:04.375] INFO:     Finished server process [7523]


In [ ]:
# Add log_config=None to prevent Uvicorn calling logging.config.dictConfig() on startup
async with local_server(log_level='trace', log_config=None, quiet=False) as test_srv:
    app = test_srv.app

>>> started: True
COMMAND  PID    USER   FD   TYPE    DEVICE SIZE/OFF NODE NAME
python  7523 solveit   41u  IPv4 169881341      0t0  TCP *:8000 (LISTEN)


<<< after stop: port 8000 free


`local_server` is an async context manager that provides a **temporary FastHTML/uvicorn server** for testing purposes, so you can access `.app`, add routes, make test requests, etc., inside the `async with` block.

It reuses an existing server if the port is already in use — finds it via `find_server()` and yields it. Raises if the port is occupied by something unknown. If the port is free, spins new server (async) using `JupyUvi` with whatever app and kwargs you pass (defaulting to a bare `FastHTML()`).

`after_start` and `before_stop` callbacks are optional hooks to let you seed state or do cleanup around the test.

After the block exits, it cleanly tears down the server (but only if *it* started it — borrowed servers are left alone), waiting up to `timeout` seconds for the port to free, with a warning if it times out.

When `quiet=False`, `local_server` optionally logs startup/shutdown diagnostics (port/process info via `lsof`).

The `log_config=None` kwarg passed in the test message prevents uvicorn from overwriting the custom `LoggerHandler` logging setup with its own `dictConfig` on startup.

In [ ]:
test_is(is_serving(test_srv), False)
test_eq(find_server(), (None, None))

Anything else?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Two small things before packaging feels complete:

1. Add the explanatory note you just asked for, near the exported `_user_ws_js` / `UserWS` cells.

2. Add a minimal usage example cell, preferably not exported:

```python
async def ws_demo(msg:str, send):
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))

uws = UserWS(app, srv)
wid,path = await uws.start('demo-ws', '/demo-ws', ws_demo)
await uws.send_client(wid, dict(msg='world'))
```

One possible final polish: rename `wid` consistently in the docs/example as “websocket initiator id”, since it’s really the DOM element id that the JS watches.

<details class='token-usage-details' markdown='1'><summary>$0.0100</summary>

`total=65,901 | in=65,733 | out=168 | cached=91.7% | $0.0100 | gpt-5.5`

</details>


Perhaps document briefly the python handlers just noting they becomes normal fasthtml WS handlers (thus returning oob).

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, good point. I’d add one short paragraph to the note:

```markdown
Python handlers passed to `UserWS.start()` are normal FastHTML websocket handlers registered with `app.ws(...)`. They receive the usual parsed parameters plus `send`, and can return/send normal FastHTML responses, including OOB swaps such as `Div(..., hx_swap_oob='true')`. `UserWS` only manages connection setup, event isolation, and access to the server-side `send` callable; it does not change FastHTML websocket handler semantics.
```

<details class='token-usage-details' markdown='1'><summary>$0.0100</summary>

`total=66,093 | in=65,978 | out=115 | cached=85.2% | $0.0100 | gpt-5.5`

</details>


No, responses from FastHTML websocket handlers are **always** traeted as OOB swaps, no need to use `hx_swap_oob`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good correction. The note should say:

```markdown
Python handlers passed to `UserWS.start()` are normal FastHTML websocket handlers registered with `app.ws(...)`. They receive the usual parsed parameters plus `send`, and can send normal FastHTML responses. FastHTML websocket responses are always treated as OOB swaps, so no `hx_swap_oob` is needed. `UserWS` only manages connection setup, event isolation, and access to the server-side `send` callable; it does not change FastHTML websocket handler semantics.
```

<details class='token-usage-details' markdown='1'><summary>$0.0100</summary>

`total=66,290 | in=66,178 | out=112 | cached=98.8% | $0.0100 | gpt-5.5`

</details>


Do you know about `pyskills`, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. `pyskills` is available here and is useful for discovering and inspecting helper modules/functions.

The usual pattern is:

```python
list_pyskills()
```

to see registered skills, then:

```python
import some.skill
doc(some.skill)
```

and before using a particular function:

```python
doc(some_func)
```

That’s especially useful here because some helper functions are async, and `doc()` makes the signature clear before calling them.

<details class='token-usage-details' markdown='1'><summary>$0.0100</summary>

`total=66,460 | in=66,358 | out=102 | cached=97.8% | $0.0100 | gpt-5.5`

</details>


In [ ]:
list_pyskills()

<unknown>:16: SyntaxWarning: invalid escape sequence '\s'


{'pyskills.edit': 'Functions for creating, viewing, and modifying files. Each editing operation returns unified diffs showing what changed.',
 'pyskills.ipynb': 'Functions for view/modifying ipynb file notebook cells. Each operation returns unified diffs showing what changed.',
 'pyskills.skill': 'Pyskills is a plugin system allowing Python packages to register "skills" (units of LLM-usable functionality) via standard Python entry points. An LLM host (e.g. solveit) discovers available pyskills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen pyskills into context using standard imports.',
 'rgapi.skill': 'Fast and flexible file discovery and search for Python. Use this when code needs `fd`-style file finding or `rg`-style searching.',
 'dialoghelper.solveitskill': 'Read, search, edit, and manage Solveit dialogs using dialoghelper.core, including dialog/message addressing, line-numbered inspection, targeted message edits, add/update

In [ ]:
doc(UserWS)

class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles"""
    def __init__(self, app, srv, js='user_ws'): ...
    def add(self, wid, path, handler): ...  # Register websocket route once
    def call(self, fn, *args): ...  # Call a function on the browser-side websocket helper
    def cleanup(self, wid): ...
    def conn(self, wid): ...  # Return FastHTML connect hook storing server send callable
    def create(self, wid, path): ...
    def disconn(self, wid): ...  # Return FastHTML disconnect hook clearing server send callable
    def send(self, wid, o): ...
    def send_client(self, wid, data): ...
    def start(self, wid, path, handler, wait=True): ...  # Register route, create browser initiator, and optionally wait for connection
    def stop(self, wid): ...  # Remove browser initiator and forget server send callable
    def url(self, path): ...  # Return public websocket URL for `path`
    def wait(self, wid): ...

Let's try creating a `pyskills` for our UserWS machinery.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea. First small step: inspect the pyskills creation helper so we follow the expected registration shape instead of guessing.

```python
from pyskills import createskill
doc(createskill)
```

After that, I’d expect we’ll make a tiny skill module that exposes just the reusable surface:

```python
__all__ = ['setup_user_ws', 'UserWS']
```

The skill docstring should explain the key context future us needs: this is for Solveit-safe user HTMX websockets, with JS injected once and `UserWS` instances created as needed.

<details class='token-usage-details' markdown='1'><summary>$0.0106</summary>

`total=68,051 | in=67,463 | out=588 | cached=9.7% | reasoning=462 | $0.0106 | gpt-5.5`

</details>


In theory, you should be able to do that yourself using `pyskills` progressive discovery and `pyrun`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'll inspect the pyskills creation helper first, then use the documented workflow rather than guessing the file layout.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pyskills import createskill\ndoc(cre…&quot;)→&quot;How to create a pyskills pyskill module.…&quot;</code></summary>

```json
{
  "id": "fc_09e6992f5cc6b290016a52238ee89c8195993d3306e50b8f38",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pyskills import createskill\ndoc(createskill)"
    }
  },
  "result": "How to create a pyskills pyskill module.\n\nA pyskill is a standard Python module that registers itself via entry points so LLM hosts can discover and load it.\n\n## 1. Create your module\n\nYour module needs:\n- A docstring: first paragraph is the short description shown during discovery; the rest is read by the LLM after loading.\n- `__all__` (optional): if provided, `doc()` and `xdir()` show exactly these symbols. Otherwise they fall back to non-private names defined in the module (not just imported), plus explicitly imported sibling submodules.\n- `allow()` calls: declares what the LLM is permitted to call.\n\n## 2. Register via entry point\n\nIn your `pyproject.toml`:\n\n    [project.entry-points.pyskills]\n    my_skill = \"mypackage.mymodule\"\n\nThe key is an arbitrary name; the value is the module path.\n\n## 3. Use `allow()` to declare permissions\n\n`allow()` populates `__pytools__`, a `defaultdict(set)` mapping modules/classes to permitted callable names. The LLM host enforces these at call time.\n\n    from pyskills.core import allow\n\n    allow(func1, func2)                          # register standalone functions\n    allow({SomeClass: ['method1', 'method2']})    # register specific methods\n    allow({SomeClass: ...})                       # register all public methods\n\n## 4. Allow policies for write-guarded operations\n\nFor operations that write to the filesystem, attach an `AllowPolicy` to constrain calls by destination path:\n\n    from pyskills.core import allow, PosAllowPolicy, PathWritePolicy, OpenWritePolicy\n\n    allow(my_write_func, allow_policy=PosAllowPolicy(0))\n\nThree built-in policies:\n- `PosAllowPolicy(pos, kw)`: check that a positional or keyword arg is under allowed destination paths.\n- `PathWritePolicy(target_pos, target_kw)`: check the Path object itself, and optionally a target arg.\n- `OpenWritePolicy()`: check open() calls only when mode includes w/a/x/+.\n\nPolicies are stored as `(name, AllowPolicy)` tuples in `__pytools__`. The host passes its allowed destination list at call time.\n\n## 5. Module contract summary\n\n    '''Short description for discovery.\n\n    Detailed docs read by the LLM after import.\n    '''\n\n    from pyskills.core import allow\n\n    __all__ = ['my_func', 'MyClass']\n\n    def my_func(x: int) -> str:\n        \"Does something useful\"\n        ...\n\n    class MyClass:\n        \"A useful class\"\n        def method(self) -> str:\n            \"Does something\"\n            ...\n\n    allow(my_func, {MyClass: ...})\n\nAfter import, the LLM inspects the module with `doc(module)` (overview of classes, functions, submodules, and allows) and `xdir(module)` (filtered list of public symbols).\n\n## 6. Local pyskills without packaging\n\nThe entry point approach above requires a full package install. For quick personal pyskills, or pyskills shared across projects with isolated environments (e.g. separate uv venvs), pyskills provides an XDG-based pyskills directory.\n\nWhen you first `import pyskills`, it creates a directory at your XDG data home (typically `~/.local/share/pyskills/`) and writes a `.pth` file into `site-packages`. This `.pth` file tells Python to add the pyskills directory to `sys.path` on startup, so any modules placed there are importable as standard Python modules without any special import machinery. This works across all Python environments on your system, even separate uv projects with isolated venvs.\n\nYou can create a pyskill programmatically with `register_pyskill`:\n\n    from pyskills.core import register_pyskill\n\n    register_pyskill('my_local.skill', 'A quick local pyskill.', code='''\n    from pyskills.core import allow\n\n    __all__ = ['hello']\n\n    def hello(name: str) -> str:\n        \"Greet someone\"\n        return f\"Hello, {name}!\"\n\n    allow(hello)\n    ''')\n\nThis writes the module file into the XDG pyskills directory and creates a minimal dist-info entry point, so the pyskill immediately appears in `list_pyskills()`.\n\nUse `enable_pyskill(name)` / `disable_pyskill(name)` to toggle a pyskill's visibility without deleting files. Use `pyskills_dir()` to see where the directory is."
}
```

</details>

The docs show `register_pyskill` is the right path for a local skill. I'll inspect that function before using it.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pyskills.core import register_pyski…&quot;)→&quot;def register_pyskill(\n    name, docstr, …&quot;</code></summary>

```json
{
  "id": "fc_053884e77e78f91d016a522391ea6481949f65a06fcf98ba0d",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pyskills.core import register_pyskill\ndoc(register_pyskill)"
    }
  },
  "result": "def register_pyskill(\n    name, docstr, code:str=''\n):\n\"\"\"Register a pyskill module `name` in the xdg pyskills dir\"\"\""
}
```

</details>

Now I'll create a local pyskill module that exposes the existing `setup_user_ws` and `UserWS` from this notebook's exported module. Since this notebook is `#| default_exp server`, the import target should be `pote.server`.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from pyskills.core import register_pyski…&quot;)→&quot;PermissionError: os.mkdir &#x27;/app/data/.lo…&quot;</code></summary>

```json
{
  "id": "fc_02b443b6753a94f7016a52239617dc8196af01cdfe5fb602e3",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from pyskills.core import register_pyskill\nregister_pyskill('pote.userws_skill', 'Solveit-safe HTMX user websocket helpers.', code=r'''\n\"\"\"Solveit-safe HTMX user websocket helpers.\n\nUse this skill for user-defined HTMX websockets inside Solveit. Inject the browser helper once with `setup_user_ws()`, then create `UserWS(app, srv)` instances as needed. `UserWS.start()` registers a normal FastHTML websocket handler, creates the browser initiator, and optionally waits for the connection. FastHTML websocket responses are treated as OOB swaps.\n\"\"\"\n\nfrom pyskills.core import allow\nfrom pote.server import setup_user_ws, UserWS\n\n__all__ = ['setup_user_ws', 'UserWS']\n\nallow(setup_user_ws, {UserWS: ...})\n''')\nlist_pyskills().get('pote.userws_skill')"
    }
  },
  "result": "PermissionError: os.mkdir '/app/data/.local/share/pyskills/pote' not in (). Code message has been added to dialog. Ask user to run it."
}
```

</details>

I inspected the pyskills creation docs and found the intended path:

- use a standard module with a docstring, `__all__`, and `allow(...)`
- for quick local skills, use `register_pyskill(name, docstr, code=...)`
- `register_pyskill` writes into the local pyskills directory and registers an entry point so it appears in `list_pyskills()`

I then tried to register a local skill:

```python
pote.userws_skill
```

intended to expose:

```python
setup_user_ws
UserWS
```

from:

```python
pote.server
```

with this public surface:

```python
__all__ = ['setup_user_ws', 'UserWS']
allow(setup_user_ws, {UserWS: ...})
```

But registration failed with a permission error while trying to create the nested package directory:

```python
PermissionError: os.mkdir '/app/data/.local/share/pyskills/pote' not in ()
```

So the skill was not created yet. The next step is to either:

1. create/register a flat local skill name that doesn’t require making a nested directory, e.g. `pote_userws_skill`, or  
2. have you run the generated registration code directly if the notebook kernel has the needed write permission outside my restricted tool call context.

<details class='token-usage-details' markdown='1'><summary>$0.0415</summary>

`total=272,561 | in=271,588 | out=973 | cached=52.5% | reasoning=287 | $0.0415 | gpt-5.5`

</details>


In [ ]:
delete_pyskill('pote.userws_skill')
clear_mod('pote.userws_skill')

In [ ]:
_doc = """Solveit-safe HTMX user websocket helpers.

Use this skill for user-defined HTMX websockets inside Solveit. Inject the browser helper once with `setup_user_ws()`, then create `UserWS(app, srv)` instances as needed. `UserWS.start()` registers a normal FastHTML websocket handler, creates the browser initiator, and optionally waits for the connection. FastHTML websocket responses are treated as OOB swaps.
"""
register_pyskill('pote.userws_skill', _doc, code=r'''
from pyskills.core import allow
from pote.server import setup_user_ws, UserWS

__all__ = ['setup_user_ws', 'UserWS']

allow(setup_user_ws, {UserWS: ...})
''')
list_pyskills().get('pote.userws_skill')

'Solveit-safe HTMX user websocket helpers.'

In [ ]:
list_pyskills()

{'pote.userws_skill': 'Solveit-safe HTMX user websocket helpers.',
 'pyskills.edit': 'Functions for creating, viewing, and modifying files. Each editing operation returns unified diffs showing what changed.',
 'pyskills.ipynb': 'Functions for view/modifying ipynb file notebook cells. Each operation returns unified diffs showing what changed.',
 'pyskills.skill': 'Pyskills is a plugin system allowing Python packages to register "skills" (units of LLM-usable functionality) via standard Python entry points. An LLM host (e.g. solveit) discovers available pyskills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen pyskills into context using standard imports.',
 'rgapi.skill': 'Fast and flexible file discovery and search for Python. Use this when code needs `fd`-style file finding or `rg`-style searching.',
 'dialoghelper.solveitskill': 'Read, search, edit, and manage Solveit dialogs using dialoghelper.core, including dialog/message addre

In [ ]:
import pote.userws_skill
doc(pote.userws_skill)

# module pote.userws_skill:

"""Solveit-safe HTMX user websocket helpers.

Use this skill for user-defined HTMX websockets inside Solveit. Inject the browser helper once with `setup_user_ws()`, then create `UserWS(app, srv)` instances as needed. `UserWS.start()` registers a normal FastHTML websocket handler, creates the browser initiator, and optionally waits for the connection. FastHTML websocket responses are treated as OOB swaps.
"""

## types:
- class UserWS: ...  # Manage Solveit-safe HTMX websocket initiators and server send handles

## functions:
- async def setup_user_ws()

In [ ]:
doc(pote.userws_skill.setup_user_ws)

async def setup_user_ws(
    
):
"""Call self as a function."""

In [ ]:
doc(pote.userws_skill.UserWS)

class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles"""
    def __init__(self, app, srv, js='user_ws'): ...
    def add(self, wid, path, handler): ...  # Register websocket route once
    def call(self, fn, *args): ...  # Call a function on the browser-side websocket helper
    def cleanup(self, wid): ...
    def conn(self, wid): ...  # Return FastHTML connect hook storing server send callable
    def create(self, wid, path): ...
    def disconn(self, wid): ...  # Return FastHTML disconnect hook clearing server send callable
    def send(self, wid, o): ...
    def send_client(self, wid, data): ...
    def start(self, wid, path, handler, wait=True): ...  # Register route, create browser initiator, and optionally wait for connection
    def stop(self, wid): ...  # Remove browser initiator and forget server send callable
    def url(self, path): ...  # Return public websocket URL for `path`
    def wait(self, wid): ...

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose" markdown="1">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()